# 🐧 Esercitazione 2 — Regressione, frontiere decisionali e alberi "sotto il cofano"

**new ALF – Tutorato** · Corso di Laurea in Informatica · AA 2026/27

Nell'Esercitazione 1 abbiamo scritto un classificatore a mano, lo abbiamo valutato e abbiamo addestrato i primi alberi di decisione con scikit-learn. Oggi facciamo un passo avanti: vediamo **che tipo di problemi** si possono risolvere, la **regressione lineare**, la **frontiera decisionale** e cosa succede **dentro** un albero quando chiamiamo `.fit()`. Chiudiamo con gli **ensemble**.

Lo faremo con un unico filo conduttore: una ricercatrice della **stazione Palmer, in Antartide**, ci ha mandato le misure di 344 pinguini e ci fa una serie di richieste.


> ### ⚠️ Prima di tutto: fai la tua copia
> Questo notebook è condiviso in **sola lettura**: puoi eseguirlo e modificarlo, ma le modifiche **non verranno salvate**.
> Vai su **File → Salva una copia in Drive**. Si aprirà una nuova scheda con la *tua* copia (la trovi poi su Google Drive, nella cartella *Colab Notebooks*): lavora lì.

### Come si usa questo notebook
- Ogni **cella** si esegue con **Shift + Invio** (o con il tasto ▶ a sinistra). Esegui le celle **in ordine**, dall'alto verso il basso.
- Le celle segnate con **✏️** sono esercizi: modifica il codice dove indicato. Sotto ogni esercizio c'è un blocco **💡 Soluzione** chiuso: prova prima da solo. Se non ci riesci, **copia la soluzione nella cella** prima di andare avanti, perché le celle successive la usano.
- I box **🐍** spiegano le cose nuove di Python che incontriamo strada facendo.
- Le sezioni segnate con **⏩** sono approfondimenti: se il tempo stringe si possono saltare senza perdere il filo.
- Se qualcosa si rompe: menu **Runtime → Riavvia sessione ed esegui tutto** (*Restart session and run all*).

In [ ]:
#@title ⚙️ Setup: esegui questa cella per prima (crea il file con i dati)
# Questa cella scrive nel workspace di Colab il file "pinguini.csv". Puoi vederlo nel pannello 📁 a sinistra.
# Nota: il workspace di Colab si svuota quando la sessione termina,
# quindi questa cella va rieseguita ogni volta che riapri il notebook.
#
# Dati: Palmer Penguins (Gorman, Williams & Fraser, 2014), raccolti alla Palmer Station (Antartide)
# e distribuiti con licenza CC0 dal pacchetto palmerpenguins di A. Horst, A. Hill e K. Gorman.

PINGUINI_CSV = """species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
Adelie,Torgersen,39.1,18.7,181,3750,MALE
Adelie,Torgersen,39.5,17.4,186,3800,FEMALE
Adelie,Torgersen,40.3,18,195,3250,FEMALE
Adelie,Torgersen,,,,,
Adelie,Torgersen,36.7,19.3,193,3450,FEMALE
Adelie,Torgersen,39.3,20.6,190,3650,MALE
Adelie,Torgersen,38.9,17.8,181,3625,FEMALE
Adelie,Torgersen,39.2,19.6,195,4675,MALE
Adelie,Torgersen,34.1,18.1,193,3475,
Adelie,Torgersen,42,20.2,190,4250,
Adelie,Torgersen,37.8,17.1,186,3300,
Adelie,Torgersen,37.8,17.3,180,3700,
Adelie,Torgersen,41.1,17.6,182,3200,FEMALE
Adelie,Torgersen,38.6,21.2,191,3800,MALE
Adelie,Torgersen,34.6,21.1,198,4400,MALE
Adelie,Torgersen,36.6,17.8,185,3700,FEMALE
Adelie,Torgersen,38.7,19,195,3450,FEMALE
Adelie,Torgersen,42.5,20.7,197,4500,MALE
Adelie,Torgersen,34.4,18.4,184,3325,FEMALE
Adelie,Torgersen,46,21.5,194,4200,MALE
Adelie,Biscoe,37.8,18.3,174,3400,FEMALE
Adelie,Biscoe,37.7,18.7,180,3600,MALE
Adelie,Biscoe,35.9,19.2,189,3800,FEMALE
Adelie,Biscoe,38.2,18.1,185,3950,MALE
Adelie,Biscoe,38.8,17.2,180,3800,MALE
Adelie,Biscoe,35.3,18.9,187,3800,FEMALE
Adelie,Biscoe,40.6,18.6,183,3550,MALE
Adelie,Biscoe,40.5,17.9,187,3200,FEMALE
Adelie,Biscoe,37.9,18.6,172,3150,FEMALE
Adelie,Biscoe,40.5,18.9,180,3950,MALE
Adelie,Dream,39.5,16.7,178,3250,FEMALE
Adelie,Dream,37.2,18.1,178,3900,MALE
Adelie,Dream,39.5,17.8,188,3300,FEMALE
Adelie,Dream,40.9,18.9,184,3900,MALE
Adelie,Dream,36.4,17,195,3325,FEMALE
Adelie,Dream,39.2,21.1,196,4150,MALE
Adelie,Dream,38.8,20,190,3950,MALE
Adelie,Dream,42.2,18.5,180,3550,FEMALE
Adelie,Dream,37.6,19.3,181,3300,FEMALE
Adelie,Dream,39.8,19.1,184,4650,MALE
Adelie,Dream,36.5,18,182,3150,FEMALE
Adelie,Dream,40.8,18.4,195,3900,MALE
Adelie,Dream,36,18.5,186,3100,FEMALE
Adelie,Dream,44.1,19.7,196,4400,MALE
Adelie,Dream,37,16.9,185,3000,FEMALE
Adelie,Dream,39.6,18.8,190,4600,MALE
Adelie,Dream,41.1,19,182,3425,MALE
Adelie,Dream,37.5,18.9,179,2975,
Adelie,Dream,36,17.9,190,3450,FEMALE
Adelie,Dream,42.3,21.2,191,4150,MALE
Adelie,Biscoe,39.6,17.7,186,3500,FEMALE
Adelie,Biscoe,40.1,18.9,188,4300,MALE
Adelie,Biscoe,35,17.9,190,3450,FEMALE
Adelie,Biscoe,42,19.5,200,4050,MALE
Adelie,Biscoe,34.5,18.1,187,2900,FEMALE
Adelie,Biscoe,41.4,18.6,191,3700,MALE
Adelie,Biscoe,39,17.5,186,3550,FEMALE
Adelie,Biscoe,40.6,18.8,193,3800,MALE
Adelie,Biscoe,36.5,16.6,181,2850,FEMALE
Adelie,Biscoe,37.6,19.1,194,3750,MALE
Adelie,Biscoe,35.7,16.9,185,3150,FEMALE
Adelie,Biscoe,41.3,21.1,195,4400,MALE
Adelie,Biscoe,37.6,17,185,3600,FEMALE
Adelie,Biscoe,41.1,18.2,192,4050,MALE
Adelie,Biscoe,36.4,17.1,184,2850,FEMALE
Adelie,Biscoe,41.6,18,192,3950,MALE
Adelie,Biscoe,35.5,16.2,195,3350,FEMALE
Adelie,Biscoe,41.1,19.1,188,4100,MALE
Adelie,Torgersen,35.9,16.6,190,3050,FEMALE
Adelie,Torgersen,41.8,19.4,198,4450,MALE
Adelie,Torgersen,33.5,19,190,3600,FEMALE
Adelie,Torgersen,39.7,18.4,190,3900,MALE
Adelie,Torgersen,39.6,17.2,196,3550,FEMALE
Adelie,Torgersen,45.8,18.9,197,4150,MALE
Adelie,Torgersen,35.5,17.5,190,3700,FEMALE
Adelie,Torgersen,42.8,18.5,195,4250,MALE
Adelie,Torgersen,40.9,16.8,191,3700,FEMALE
Adelie,Torgersen,37.2,19.4,184,3900,MALE
Adelie,Torgersen,36.2,16.1,187,3550,FEMALE
Adelie,Torgersen,42.1,19.1,195,4000,MALE
Adelie,Torgersen,34.6,17.2,189,3200,FEMALE
Adelie,Torgersen,42.9,17.6,196,4700,MALE
Adelie,Torgersen,36.7,18.8,187,3800,FEMALE
Adelie,Torgersen,35.1,19.4,193,4200,MALE
Adelie,Dream,37.3,17.8,191,3350,FEMALE
Adelie,Dream,41.3,20.3,194,3550,MALE
Adelie,Dream,36.3,19.5,190,3800,MALE
Adelie,Dream,36.9,18.6,189,3500,FEMALE
Adelie,Dream,38.3,19.2,189,3950,MALE
Adelie,Dream,38.9,18.8,190,3600,FEMALE
Adelie,Dream,35.7,18,202,3550,FEMALE
Adelie,Dream,41.1,18.1,205,4300,MALE
Adelie,Dream,34,17.1,185,3400,FEMALE
Adelie,Dream,39.6,18.1,186,4450,MALE
Adelie,Dream,36.2,17.3,187,3300,FEMALE
Adelie,Dream,40.8,18.9,208,4300,MALE
Adelie,Dream,38.1,18.6,190,3700,FEMALE
Adelie,Dream,40.3,18.5,196,4350,MALE
Adelie,Dream,33.1,16.1,178,2900,FEMALE
Adelie,Dream,43.2,18.5,192,4100,MALE
Adelie,Biscoe,35,17.9,192,3725,FEMALE
Adelie,Biscoe,41,20,203,4725,MALE
Adelie,Biscoe,37.7,16,183,3075,FEMALE
Adelie,Biscoe,37.8,20,190,4250,MALE
Adelie,Biscoe,37.9,18.6,193,2925,FEMALE
Adelie,Biscoe,39.7,18.9,184,3550,MALE
Adelie,Biscoe,38.6,17.2,199,3750,FEMALE
Adelie,Biscoe,38.2,20,190,3900,MALE
Adelie,Biscoe,38.1,17,181,3175,FEMALE
Adelie,Biscoe,43.2,19,197,4775,MALE
Adelie,Biscoe,38.1,16.5,198,3825,FEMALE
Adelie,Biscoe,45.6,20.3,191,4600,MALE
Adelie,Biscoe,39.7,17.7,193,3200,FEMALE
Adelie,Biscoe,42.2,19.5,197,4275,MALE
Adelie,Biscoe,39.6,20.7,191,3900,FEMALE
Adelie,Biscoe,42.7,18.3,196,4075,MALE
Adelie,Torgersen,38.6,17,188,2900,FEMALE
Adelie,Torgersen,37.3,20.5,199,3775,MALE
Adelie,Torgersen,35.7,17,189,3350,FEMALE
Adelie,Torgersen,41.1,18.6,189,3325,MALE
Adelie,Torgersen,36.2,17.2,187,3150,FEMALE
Adelie,Torgersen,37.7,19.8,198,3500,MALE
Adelie,Torgersen,40.2,17,176,3450,FEMALE
Adelie,Torgersen,41.4,18.5,202,3875,MALE
Adelie,Torgersen,35.2,15.9,186,3050,FEMALE
Adelie,Torgersen,40.6,19,199,4000,MALE
Adelie,Torgersen,38.8,17.6,191,3275,FEMALE
Adelie,Torgersen,41.5,18.3,195,4300,MALE
Adelie,Torgersen,39,17.1,191,3050,FEMALE
Adelie,Torgersen,44.1,18,210,4000,MALE
Adelie,Torgersen,38.5,17.9,190,3325,FEMALE
Adelie,Torgersen,43.1,19.2,197,3500,MALE
Adelie,Dream,36.8,18.5,193,3500,FEMALE
Adelie,Dream,37.5,18.5,199,4475,MALE
Adelie,Dream,38.1,17.6,187,3425,FEMALE
Adelie,Dream,41.1,17.5,190,3900,MALE
Adelie,Dream,35.6,17.5,191,3175,FEMALE
Adelie,Dream,40.2,20.1,200,3975,MALE
Adelie,Dream,37,16.5,185,3400,FEMALE
Adelie,Dream,39.7,17.9,193,4250,MALE
Adelie,Dream,40.2,17.1,193,3400,FEMALE
Adelie,Dream,40.6,17.2,187,3475,MALE
Adelie,Dream,32.1,15.5,188,3050,FEMALE
Adelie,Dream,40.7,17,190,3725,MALE
Adelie,Dream,37.3,16.8,192,3000,FEMALE
Adelie,Dream,39,18.7,185,3650,MALE
Adelie,Dream,39.2,18.6,190,4250,MALE
Adelie,Dream,36.6,18.4,184,3475,FEMALE
Adelie,Dream,36,17.8,195,3450,FEMALE
Adelie,Dream,37.8,18.1,193,3750,MALE
Adelie,Dream,36,17.1,187,3700,FEMALE
Adelie,Dream,41.5,18.5,201,4000,MALE
Chinstrap,Dream,46.5,17.9,192,3500,FEMALE
Chinstrap,Dream,50,19.5,196,3900,MALE
Chinstrap,Dream,51.3,19.2,193,3650,MALE
Chinstrap,Dream,45.4,18.7,188,3525,FEMALE
Chinstrap,Dream,52.7,19.8,197,3725,MALE
Chinstrap,Dream,45.2,17.8,198,3950,FEMALE
Chinstrap,Dream,46.1,18.2,178,3250,FEMALE
Chinstrap,Dream,51.3,18.2,197,3750,MALE
Chinstrap,Dream,46,18.9,195,4150,FEMALE
Chinstrap,Dream,51.3,19.9,198,3700,MALE
Chinstrap,Dream,46.6,17.8,193,3800,FEMALE
Chinstrap,Dream,51.7,20.3,194,3775,MALE
Chinstrap,Dream,47,17.3,185,3700,FEMALE
Chinstrap,Dream,52,18.1,201,4050,MALE
Chinstrap,Dream,45.9,17.1,190,3575,FEMALE
Chinstrap,Dream,50.5,19.6,201,4050,MALE
Chinstrap,Dream,50.3,20,197,3300,MALE
Chinstrap,Dream,58,17.8,181,3700,FEMALE
Chinstrap,Dream,46.4,18.6,190,3450,FEMALE
Chinstrap,Dream,49.2,18.2,195,4400,MALE
Chinstrap,Dream,42.4,17.3,181,3600,FEMALE
Chinstrap,Dream,48.5,17.5,191,3400,MALE
Chinstrap,Dream,43.2,16.6,187,2900,FEMALE
Chinstrap,Dream,50.6,19.4,193,3800,MALE
Chinstrap,Dream,46.7,17.9,195,3300,FEMALE
Chinstrap,Dream,52,19,197,4150,MALE
Chinstrap,Dream,50.5,18.4,200,3400,FEMALE
Chinstrap,Dream,49.5,19,200,3800,MALE
Chinstrap,Dream,46.4,17.8,191,3700,FEMALE
Chinstrap,Dream,52.8,20,205,4550,MALE
Chinstrap,Dream,40.9,16.6,187,3200,FEMALE
Chinstrap,Dream,54.2,20.8,201,4300,MALE
Chinstrap,Dream,42.5,16.7,187,3350,FEMALE
Chinstrap,Dream,51,18.8,203,4100,MALE
Chinstrap,Dream,49.7,18.6,195,3600,MALE
Chinstrap,Dream,47.5,16.8,199,3900,FEMALE
Chinstrap,Dream,47.6,18.3,195,3850,FEMALE
Chinstrap,Dream,52,20.7,210,4800,MALE
Chinstrap,Dream,46.9,16.6,192,2700,FEMALE
Chinstrap,Dream,53.5,19.9,205,4500,MALE
Chinstrap,Dream,49,19.5,210,3950,MALE
Chinstrap,Dream,46.2,17.5,187,3650,FEMALE
Chinstrap,Dream,50.9,19.1,196,3550,MALE
Chinstrap,Dream,45.5,17,196,3500,FEMALE
Chinstrap,Dream,50.9,17.9,196,3675,FEMALE
Chinstrap,Dream,50.8,18.5,201,4450,MALE
Chinstrap,Dream,50.1,17.9,190,3400,FEMALE
Chinstrap,Dream,49,19.6,212,4300,MALE
Chinstrap,Dream,51.5,18.7,187,3250,MALE
Chinstrap,Dream,49.8,17.3,198,3675,FEMALE
Chinstrap,Dream,48.1,16.4,199,3325,FEMALE
Chinstrap,Dream,51.4,19,201,3950,MALE
Chinstrap,Dream,45.7,17.3,193,3600,FEMALE
Chinstrap,Dream,50.7,19.7,203,4050,MALE
Chinstrap,Dream,42.5,17.3,187,3350,FEMALE
Chinstrap,Dream,52.2,18.8,197,3450,MALE
Chinstrap,Dream,45.2,16.6,191,3250,FEMALE
Chinstrap,Dream,49.3,19.9,203,4050,MALE
Chinstrap,Dream,50.2,18.8,202,3800,MALE
Chinstrap,Dream,45.6,19.4,194,3525,FEMALE
Chinstrap,Dream,51.9,19.5,206,3950,MALE
Chinstrap,Dream,46.8,16.5,189,3650,FEMALE
Chinstrap,Dream,45.7,17,195,3650,FEMALE
Chinstrap,Dream,55.8,19.8,207,4000,MALE
Chinstrap,Dream,43.5,18.1,202,3400,FEMALE
Chinstrap,Dream,49.6,18.2,193,3775,MALE
Chinstrap,Dream,50.8,19,210,4100,MALE
Chinstrap,Dream,50.2,18.7,198,3775,FEMALE
Gentoo,Biscoe,46.1,13.2,211,4500,FEMALE
Gentoo,Biscoe,50,16.3,230,5700,MALE
Gentoo,Biscoe,48.7,14.1,210,4450,FEMALE
Gentoo,Biscoe,50,15.2,218,5700,MALE
Gentoo,Biscoe,47.6,14.5,215,5400,MALE
Gentoo,Biscoe,46.5,13.5,210,4550,FEMALE
Gentoo,Biscoe,45.4,14.6,211,4800,FEMALE
Gentoo,Biscoe,46.7,15.3,219,5200,MALE
Gentoo,Biscoe,43.3,13.4,209,4400,FEMALE
Gentoo,Biscoe,46.8,15.4,215,5150,MALE
Gentoo,Biscoe,40.9,13.7,214,4650,FEMALE
Gentoo,Biscoe,49,16.1,216,5550,MALE
Gentoo,Biscoe,45.5,13.7,214,4650,FEMALE
Gentoo,Biscoe,48.4,14.6,213,5850,MALE
Gentoo,Biscoe,45.8,14.6,210,4200,FEMALE
Gentoo,Biscoe,49.3,15.7,217,5850,MALE
Gentoo,Biscoe,42,13.5,210,4150,FEMALE
Gentoo,Biscoe,49.2,15.2,221,6300,MALE
Gentoo,Biscoe,46.2,14.5,209,4800,FEMALE
Gentoo,Biscoe,48.7,15.1,222,5350,MALE
Gentoo,Biscoe,50.2,14.3,218,5700,MALE
Gentoo,Biscoe,45.1,14.5,215,5000,FEMALE
Gentoo,Biscoe,46.5,14.5,213,4400,FEMALE
Gentoo,Biscoe,46.3,15.8,215,5050,MALE
Gentoo,Biscoe,42.9,13.1,215,5000,FEMALE
Gentoo,Biscoe,46.1,15.1,215,5100,MALE
Gentoo,Biscoe,44.5,14.3,216,4100,
Gentoo,Biscoe,47.8,15,215,5650,MALE
Gentoo,Biscoe,48.2,14.3,210,4600,FEMALE
Gentoo,Biscoe,50,15.3,220,5550,MALE
Gentoo,Biscoe,47.3,15.3,222,5250,MALE
Gentoo,Biscoe,42.8,14.2,209,4700,FEMALE
Gentoo,Biscoe,45.1,14.5,207,5050,FEMALE
Gentoo,Biscoe,59.6,17,230,6050,MALE
Gentoo,Biscoe,49.1,14.8,220,5150,FEMALE
Gentoo,Biscoe,48.4,16.3,220,5400,MALE
Gentoo,Biscoe,42.6,13.7,213,4950,FEMALE
Gentoo,Biscoe,44.4,17.3,219,5250,MALE
Gentoo,Biscoe,44,13.6,208,4350,FEMALE
Gentoo,Biscoe,48.7,15.7,208,5350,MALE
Gentoo,Biscoe,42.7,13.7,208,3950,FEMALE
Gentoo,Biscoe,49.6,16,225,5700,MALE
Gentoo,Biscoe,45.3,13.7,210,4300,FEMALE
Gentoo,Biscoe,49.6,15,216,4750,MALE
Gentoo,Biscoe,50.5,15.9,222,5550,MALE
Gentoo,Biscoe,43.6,13.9,217,4900,FEMALE
Gentoo,Biscoe,45.5,13.9,210,4200,FEMALE
Gentoo,Biscoe,50.5,15.9,225,5400,MALE
Gentoo,Biscoe,44.9,13.3,213,5100,FEMALE
Gentoo,Biscoe,45.2,15.8,215,5300,MALE
Gentoo,Biscoe,46.6,14.2,210,4850,FEMALE
Gentoo,Biscoe,48.5,14.1,220,5300,MALE
Gentoo,Biscoe,45.1,14.4,210,4400,FEMALE
Gentoo,Biscoe,50.1,15,225,5000,MALE
Gentoo,Biscoe,46.5,14.4,217,4900,FEMALE
Gentoo,Biscoe,45,15.4,220,5050,MALE
Gentoo,Biscoe,43.8,13.9,208,4300,FEMALE
Gentoo,Biscoe,45.5,15,220,5000,MALE
Gentoo,Biscoe,43.2,14.5,208,4450,FEMALE
Gentoo,Biscoe,50.4,15.3,224,5550,MALE
Gentoo,Biscoe,45.3,13.8,208,4200,FEMALE
Gentoo,Biscoe,46.2,14.9,221,5300,MALE
Gentoo,Biscoe,45.7,13.9,214,4400,FEMALE
Gentoo,Biscoe,54.3,15.7,231,5650,MALE
Gentoo,Biscoe,45.8,14.2,219,4700,FEMALE
Gentoo,Biscoe,49.8,16.8,230,5700,MALE
Gentoo,Biscoe,46.2,14.4,214,4650,
Gentoo,Biscoe,49.5,16.2,229,5800,MALE
Gentoo,Biscoe,43.5,14.2,220,4700,FEMALE
Gentoo,Biscoe,50.7,15,223,5550,MALE
Gentoo,Biscoe,47.7,15,216,4750,FEMALE
Gentoo,Biscoe,46.4,15.6,221,5000,MALE
Gentoo,Biscoe,48.2,15.6,221,5100,MALE
Gentoo,Biscoe,46.5,14.8,217,5200,FEMALE
Gentoo,Biscoe,46.4,15,216,4700,FEMALE
Gentoo,Biscoe,48.6,16,230,5800,MALE
Gentoo,Biscoe,47.5,14.2,209,4600,FEMALE
Gentoo,Biscoe,51.1,16.3,220,6000,MALE
Gentoo,Biscoe,45.2,13.8,215,4750,FEMALE
Gentoo,Biscoe,45.2,16.4,223,5950,MALE
Gentoo,Biscoe,49.1,14.5,212,4625,FEMALE
Gentoo,Biscoe,52.5,15.6,221,5450,MALE
Gentoo,Biscoe,47.4,14.6,212,4725,FEMALE
Gentoo,Biscoe,50,15.9,224,5350,MALE
Gentoo,Biscoe,44.9,13.8,212,4750,FEMALE
Gentoo,Biscoe,50.8,17.3,228,5600,MALE
Gentoo,Biscoe,43.4,14.4,218,4600,FEMALE
Gentoo,Biscoe,51.3,14.2,218,5300,MALE
Gentoo,Biscoe,47.5,14,212,4875,FEMALE
Gentoo,Biscoe,52.1,17,230,5550,MALE
Gentoo,Biscoe,47.5,15,218,4950,FEMALE
Gentoo,Biscoe,52.2,17.1,228,5400,MALE
Gentoo,Biscoe,45.5,14.5,212,4750,FEMALE
Gentoo,Biscoe,49.5,16.1,224,5650,MALE
Gentoo,Biscoe,44.5,14.7,214,4850,FEMALE
Gentoo,Biscoe,50.8,15.7,226,5200,MALE
Gentoo,Biscoe,49.4,15.8,216,4925,MALE
Gentoo,Biscoe,46.9,14.6,222,4875,FEMALE
Gentoo,Biscoe,48.4,14.4,203,4625,FEMALE
Gentoo,Biscoe,51.1,16.5,225,5250,MALE
Gentoo,Biscoe,48.5,15,219,4850,FEMALE
Gentoo,Biscoe,55.9,17,228,5600,MALE
Gentoo,Biscoe,47.2,15.5,215,4975,FEMALE
Gentoo,Biscoe,49.1,15,228,5500,MALE
Gentoo,Biscoe,47.3,13.8,216,4725,
Gentoo,Biscoe,46.8,16.1,215,5500,MALE
Gentoo,Biscoe,41.7,14.7,210,4700,FEMALE
Gentoo,Biscoe,53.4,15.8,219,5500,MALE
Gentoo,Biscoe,43.3,14,208,4575,FEMALE
Gentoo,Biscoe,48.1,15.1,209,5500,MALE
Gentoo,Biscoe,50.5,15.2,216,5000,FEMALE
Gentoo,Biscoe,49.8,15.9,229,5950,MALE
Gentoo,Biscoe,43.5,15.2,213,4650,FEMALE
Gentoo,Biscoe,51.5,16.3,230,5500,MALE
Gentoo,Biscoe,46.2,14.1,217,4375,FEMALE
Gentoo,Biscoe,55.1,16,230,5850,MALE
Gentoo,Biscoe,44.5,15.7,217,4875,
Gentoo,Biscoe,48.8,16.2,222,6000,MALE
Gentoo,Biscoe,47.2,13.7,214,4925,FEMALE
Gentoo,Biscoe,,,,,
Gentoo,Biscoe,46.8,14.3,215,4850,FEMALE
Gentoo,Biscoe,50.4,15.7,222,5750,MALE
Gentoo,Biscoe,45.2,14.8,212,5200,FEMALE
Gentoo,Biscoe,49.9,16.1,213,5400,MALE
"""

with open("pinguini.csv", "w", encoding="utf-8") as f:
    f.write(PINGUINI_CSV)

print("✅ File pinguini.csv creato!")

In [ ]:
#@title 🛠️ Funzioni di supporto per i grafici (esegui la cella, non serve leggerla)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, to_rgb

# Un colore fisso per ogni classe, così lo stesso pinguino ha sempre lo stesso colore
COLORI = {"Adelie": "#FF8C00", "Chinstrap": "#A034F0", "Gentoo": "#159090",
          "femmina": "#D9534F", "maschio": "#2C7FB8"}

def predici(modello, X):
    """Predizioni per X: funziona sia con un modello sklearn (che ha .predict)
    sia con una funzione scritta da noi che riceve un DataFrame."""
    if hasattr(modello, "predict"):
        return np.asarray(modello.predict(X))
    return np.asarray(modello(X))

def _griglia(X, passi=300):
    c1, c2 = X.columns
    m1 = (X[c1].max() - X[c1].min()) * 0.05
    m2 = (X[c2].max() - X[c2].min()) * 0.05
    xx, yy = np.meshgrid(np.linspace(X[c1].min() - m1, X[c1].max() + m1, passi),
                         np.linspace(X[c2].min() - m2, X[c2].max() + m2, passi))
    return xx, yy, pd.DataFrame({c1: xx.ravel(), c2: yy.ravel()})

def mostra_punti(X, y, titolo="", ax=None):
    """Scatter plot di due feature, un colore per classe."""
    nuovo = ax is None
    if nuovo:
        fig, ax = plt.subplots(figsize=(7, 5))
    c1, c2 = X.columns
    for classe in sorted(pd.unique(np.asarray(y))):
        sel = np.asarray(y) == classe
        ax.scatter(X[c1][sel], X[c2][sel], c=COLORI.get(classe, "gray"), label=classe,
                   edgecolors="k", linewidths=0.5, s=35)
    ax.set_xlabel(c1); ax.set_ylabel(c2); ax.set_title(titolo)
    ax.legend(fontsize=8); ax.grid(alpha=0.3)
    if nuovo:
        plt.show()

def disegna_frontiera(modello, X, y, titolo="", ax=None):
    """Colora il piano secondo la classe predetta dal modello (frontiera decisionale)."""
    nuovo = ax is None
    if nuovo:
        fig, ax = plt.subplots(figsize=(7, 5))
    xx, yy, griglia = _griglia(X)
    pred = predici(modello, griglia)
    classi = sorted(set(pd.unique(pred)) | set(pd.unique(np.asarray(y))))
    indice = {c: i for i, c in enumerate(classi)}
    z = np.array([indice[p] for p in pred]).reshape(xx.shape)
    cmap = ListedColormap([COLORI.get(c, "gray") for c in classi])
    ax.contourf(xx, yy, z, alpha=0.25, cmap=cmap, levels=np.arange(len(classi) + 1) - 0.5)
    mostra_punti(X, y, titolo, ax=ax)
    if nuovo:
        plt.show()

def disegna_confidenza(modello, X, y, titolo="", ax=None):
    """Come disegna_frontiera, ma il colore è tanto più intenso quanto più il modello è sicuro."""
    nuovo = ax is None
    if nuovo:
        fig, ax = plt.subplots(figsize=(7, 5))
    xx, yy, griglia = _griglia(X)
    proba = modello.predict_proba(griglia)
    k = proba.shape[1]
    sicurezza = (proba.max(axis=1) - 1 / k) / (1 - 1 / k)       # 0 = indeciso, 1 = sicurissimo
    colori = np.array([to_rgb(COLORI.get(c, "gray")) for c in modello.classes_])[proba.argmax(axis=1)]
    immagine = 1 - 0.7 * sicurezza[:, None] * (1 - colori)       # sfuma verso il bianco se poco sicuro
    ax.imshow(immagine.reshape(xx.shape + (3,)), origin="lower", aspect="auto",
              extent=(xx.min(), xx.max(), yy.min(), yy.max()))
    mostra_punti(X, y, titolo, ax=ax)
    if nuovo:
        plt.show()

print("✅ Funzioni pronte: mostra_punti, disegna_frontiera, disegna_confidenza")

### 🔁 Dall'Esercitazione 1 ci portiamo dietro
- **`X`** (maiuscola) sono le **feature**, l'input del modello; **`y`** è il **target**, l'output da predire.
- I dati si dividono in **training set** (per addestrare) e **test set** (per valutare alla fine), con `train_test_split`.
- Ogni modello di scikit-learn si usa allo stesso modo: lo **creo** scegliendo gli iperparametri, lo **addestro** con `.fit(X, y)`, lo **uso** con `.predict(X)`.
- Un modello che va benissimo sul training e male sul test è in **overfitting**.

---
# Parte 1 — *"Ecco i dati: cosa ci possiamo fare?"*

> 📩 *Ciao! Vi mando le misure che abbiamo raccolto su 344 pinguini di tre specie (Adelie, Chinstrap e Gentoo), in tre isole dell'arcipelago Palmer. Per ognuno abbiamo misurato il becco, la pinna e il peso, e dove possibile abbiamo annotato il sesso. Vorremmo usare il machine learning per alleggerire il lavoro sul campo: cosa ci potete fare?*

### 1.1 Carichiamo e puliamo i dati

In [ ]:
pinguini = pd.read_csv("pinguini.csv")

# rinomino le colonne in italiano: rename riceve un dizionario {nome_vecchio: nome_nuovo}
pinguini = pinguini.rename(columns={
    "species": "specie", "island": "isola",
    "bill_length_mm": "becco_lunghezza", "bill_depth_mm": "becco_profondita",
    "flipper_length_mm": "pinna", "body_mass_g": "peso", "sex": "sesso",
})
pinguini["sesso"] = pinguini["sesso"].map({"MALE": "maschio", "FEMALE": "femmina"})

pinguini.head()

| Colonna | Significato |
|---|---|
| `specie` | Adelie, Chinstrap o Gentoo |
| `isola` | Biscoe, Dream o Torgersen |
| `becco_lunghezza`, `becco_profondita` | lunghezza e "altezza" del becco, in mm |
| `pinna` | lunghezza della pinna, in mm |
| `peso` | in grammi |
| `sesso` | maschio o femmina |

Nella quarta riga ci sono dei `NaN` (*Not a Number*): è il modo in cui pandas indica un **valore mancante**. Contiamoli.

In [ ]:
# .isna() restituisce una tabella di True/False (True = valore mancante);
# .sum() conta i True colonna per colonna (True vale 1, False vale 0)
pinguini.isna().sum()

I buchi sono di due tipi:
- **2 pinguini** non hanno **nessuna** misura: non ci servono a niente;
- **9 pinguini** hanno tutte le misure ma **non il sesso**. Questi li teniamo da parte: torneranno utili nella Parte 4.

In [ ]:
senza_misure = pinguini["peso"].isna()                       # colonna di True/False
senza_sesso = pinguini["sesso"].isna() & ~senza_misure       # & = "AND" ; ~ = "NOT"

misteriosi = pinguini[senza_sesso]     # i 9 pinguini di cui non conosciamo il sesso
df = pinguini.dropna()                 # .dropna() elimina tutte le righe con almeno un valore mancante

print("Pinguini completi:  ", len(df))
print("Pinguini senza sesso:", len(misteriosi))
df["specie"].value_counts()

> 🐍 **Filtri booleani.** `pinguini["peso"].isna()` è una colonna di `True`/`False`. Si possono combinare elemento per elemento con `&` (*AND*), `|` (*OR*) e `~` (*NOT*). Mettendo la colonna di `True`/`False` tra parentesi quadre, `pinguini[filtro]`, si tengono solo le righe in cui vale `True`.

Prima di costruire qualsiasi modello, **guardiamo i dati**. Il *pairplot* della libreria **seaborn** disegna uno scatter plot per ogni coppia di misure.

In [ ]:
import seaborn as sns    # libreria di grafici statistici costruita sopra matplotlib

sns.pairplot(df, hue="specie", palette=COLORI, height=2.2,
             vars=["becco_lunghezza", "becco_profondita", "pinna", "peso"])
plt.show()

> 🤔 Quale **coppia** di misure separa meglio le tre specie? E quale misura da sola basta a riconoscere i Gentoo?

### 1.2 Una tabella, quattro problemi 
La stessa tabella può rispondere a domande diverse. **Cambia il target `y`**, e con lui il tipo di problema:

| Domanda | Target `y` | Tipo di problema |
|---|---|---|
| *"Quanto pesa?"* | `peso`: un numero | **regressione** |
| *"È maschio o femmina?"* | `sesso`: 2 valori possibili | **classificazione binaria** |
| *"Che specie è?"* | `specie`: 3 valori possibili | **classificazione multiclasse** |
| *"Quali specie compaiono in questa foto?"* | un **insieme** di specie | **classificazione multietichetta** |

In [ ]:
target_possibili = {
    "peso": df["peso"],
    "sesso": df["sesso"],
    "specie": df["specie"],
}
# .items() scorre le coppie (chiave, valore) del dizionario
for nome, y in target_possibili.items():
    # .dtype = tipo dei valori; .nunique() = quanti valori diversi ci sono
    print(f"{nome:<7} tipo: {str(y.dtype):<8} valori diversi: {y.nunique()}")

Vediamo i primi tre problemi sullo **stesso grafico**: cambia solo il colore dei punti, cioè cosa vogliamo predire.

In [ ]:
X_becco = df[["becco_lunghezza", "becco_profondita"]]

# plt.subplots(1, 3) crea una figura con 1 riga e 3 colonne di grafici: "assi" è un array di 3 elementi
fig, assi = plt.subplots(1, 3, figsize=(17, 4.5))

# 1) regressione: il colore è un numero continuo (il peso)
punti = assi[0].scatter(X_becco["becco_lunghezza"], X_becco["becco_profondita"], c=df["peso"],
                        cmap="viridis", edgecolors="k", linewidths=0.5, s=35)
fig.colorbar(punti, ax=assi[0], label="peso (g)")
assi[0].set_xlabel("becco_lunghezza"); assi[0].set_ylabel("becco_profondita")
assi[0].set_title("Regressione: y = peso"); assi[0].grid(alpha=0.3)

# 2) classificazione binaria e 3) multiclasse: il colore è un'etichetta
mostra_punti(X_becco, df["sesso"], "Binaria: y = sesso", ax=assi[1])
mostra_punti(X_becco, df["specie"], "Multiclasse: y = specie", ax=assi[2])
plt.show()

**E la multietichetta?** Immaginiamo che la stazione abbia delle **fototrappole** sulla spiaggia. In ogni foto possono comparire **nessuna, una o più specie**: le classi **non si escludono a vicenda**.

*(Le foto sono inventate per l'esempio.)*

In [ ]:
from sklearn.preprocessing import MultiLabelBinarizer

foto = {
    "foto_01": ["Adelie"],
    "foto_02": ["Adelie", "Gentoo"],
    "foto_03": [],                                 # nessun pinguino
    "foto_04": ["Chinstrap", "Adelie", "Gentoo"],
    "foto_05": ["Gentoo"],
    "foto_06": ["Chinstrap", "Adelie"],
}

# MultiLabelBinarizer trasforma le liste di etichette in una matrice di 0 e 1: una colonna per classe
mlb = MultiLabelBinarizer()
Y_foto = pd.DataFrame(mlb.fit_transform(foto.values()), index=list(foto.keys()), columns=mlb.classes_)
Y_foto

In [ ]:
# La stessa matrice, disegnata: ogni COLONNA è un problema di classificazione BINARIA
plt.figure(figsize=(6, 4))
plt.imshow(Y_foto.to_numpy(), cmap="Greens", vmin=0, vmax=1.5, aspect="auto")
for i in range(Y_foto.shape[0]):          # .shape = (numero righe, numero colonne)
    for j in range(Y_foto.shape[1]):
        plt.text(j, i, Y_foto.iloc[i, j], ha="center", va="center")   # .iloc[riga, colonna] per posizione
plt.xticks(range(Y_foto.shape[1]), Y_foto.columns)
plt.yticks(range(Y_foto.shape[0]), Y_foto.index)
plt.title("Multietichetta: y è una matrice")
plt.show()

📌 Nella multietichetta **`y` non è più una colonna ma una matrice**. La soluzione vista a lezione è **un classificatore binario per ogni colonna** (*"c'è un Adelie? sì/no"*, *"c'è un Gentoo? sì/no"*…) e poi si mettono insieme le risposte. Molti modelli di scikit-learn, alberi compresi, accettano direttamente una matrice come `y` e lo fanno per noi.

### 🤔 Mettiti alla prova
Che tipo di problema è?
1. Stimare l'**età** di un pinguino dalle sue misure.
2. Prevedere **su quale isola** vive un pinguino.
3. Prevedere se un pinguino **supererà l'inverno**.
4. Etichettare un video con i **comportamenti** osservati (nuota, mangia, cova, litiga…).

<details>
<summary><b>💡 Risposte</b></summary>

1. **Regressione**: l'età è un numero.
2. **Multiclasse**: 3 isole possibili, e ogni pinguino vive su una sola.
3. **Binaria**: sì oppure no.
4. **Multietichetta**: nello stesso video possono comparire più comportamenti.

</details>

---
# Parte 2 — *"La bilancia si è rotta: stimate il peso"*

> 📩 *Pesare un pinguino è il lavoro più faticoso: bisogna catturarlo e metterlo in un sacco sulla bilancia. La pinna invece si misura in un attimo, anche da una foto. Potete stimare il peso a partire dalla pinna?*

Il target è un **numero**: è un problema di **regressione**.

### 2.1 Guardiamo i dati

In [ ]:
from sklearn.model_selection import train_test_split

X_peso = df[["pinna"]]      # doppie parentesi: un DataFrame con una sola colonna (sklearn vuole X "a tabella")
y_peso = df["peso"]         # parentesi singole: una colonna

X_peso_train, X_peso_test, y_peso_train, y_peso_test = train_test_split(
    X_peso, y_peso, test_size=0.3, random_state=0)

plt.figure(figsize=(7, 5))
plt.scatter(X_peso_train["pinna"], y_peso_train, edgecolors="k", linewidths=0.5, s=35)
plt.xlabel("pinna (mm)"); plt.ylabel("peso (g)"); plt.title("Training set")
plt.grid(alpha=0.3)
plt.show()

I punti stanno più o meno lungo una **retta**. Un modello lineare con una feature è proprio una retta:

$$\hat{y} = w \cdot x + b$$

dove $x$ è la pinna, $w$ la **pendenza** e $b$ l'**intercetta** (nelle slide: $w_0 + w_a x_a$). Addestrare il modello vuol dire **trovare i valori di $w$ e $b$**.

### 2.2 ✏️ Esercizio: quanto è buona una retta? L'MSE
Per confrontare due rette serve un numero. Nella regressione non si contano TP e FP: si misura **quanto sono lontane** le predizioni dai valori veri. L'**errore quadratico medio** (*Mean Squared Error*) è:

$$MSE = \frac{1}{n}\sum_{i=1}^{n}\left(y_i - \hat{y}_i\right)^2$$

Le differenze $y_i - \hat y_i$ si chiamano **residui**. Elevarle al quadrato evita che errori positivi e negativi si compensino, e pesa di più gli errori grandi.

> 🐍 **NumPy** è la libreria per il calcolo numerico. Il suo oggetto base è l'**array**, una sequenza di numeri su cui le operazioni avvengono **elemento per elemento, senza scrivere cicli `for`**. Esegui la cella qui sotto e guarda i risultati.

In [ ]:
a = np.array([3, 5, 7])
b = np.array([2, 5, 9])

print(a - b)            # [ 1  0 -2]   sottrazione elemento per elemento
print((a - b) ** 2)     # [1 0 4]      ** è l'elevamento a potenza
print(np.mean(a))       # 5.0          media
print(a * 2 + 1)        # [ 7 11 15]   anche le operazioni con un numero valgono per ogni elemento

Completa la funzione `mse` usando le operazioni NumPy qui sopra (basta una riga).

In [ ]:
def mse(y_vero, y_predetto):
    y_vero = np.asarray(y_vero)          # converte in array NumPy (accetta liste, colonne pandas...)
    y_predetto = np.asarray(y_predetto)
    # ✏️ calcola l'errore quadratico medio
    return 0

print(mse([3, 5, 7], [2, 5, 9]))   # deve stampare 1.666... cioè (1 + 0 + 4) / 3

<details>
<summary><b>💡 Soluzione</b> (clicca per aprire, ma prima prova da solo!)</summary>

```python
def mse(y_vero, y_predetto):
    y_vero = np.asarray(y_vero)
    y_predetto = np.asarray(y_predetto)
    return np.mean((y_vero - y_predetto) ** 2)

print(mse([3, 5, 7], [2, 5, 9]))
```

</details>

### 2.3 ✏️ Esercizio: sei tu la regressione
Muovi gli **slider** per cambiare $w$ e $b$: le linee grigie sono i **residui** e nel titolo vedi l'MSE sul training set. **Trova la retta con l'MSE più basso possibile** e scrivi i valori nella cella successiva.

💡 *Suggerimento*: prima scegli la pendenza $w$, poi sposta la retta su e giù con $b$.

In [ ]:
from ipywidgets import interact, FloatSlider    # ipywidgets: elementi interattivi nei notebook

def prova_retta(w=20.0, b=0.0):
    x = X_peso_train["pinna"]
    pred = w * x + b                                   # predizioni della retta: NumPy/pandas, niente for!
    plt.figure(figsize=(8, 5))
    plt.vlines(x, pred, y_peso_train, color="gray", alpha=0.4, linewidth=1)   # i residui
    plt.scatter(x, y_peso_train, edgecolors="k", linewidths=0.5, s=35)
    xs = np.array([170, 235])
    plt.plot(xs, w * xs + b, color="crimson", linewidth=2)
    plt.ylim(2000, 7000)
    plt.xlabel("pinna (mm)"); plt.ylabel("peso (g)"); plt.grid(alpha=0.3)
    plt.title(f"ŷ = {w:.1f} · pinna {b:+.0f}      MSE = {mse(y_peso_train, pred):,.0f}")
    plt.show()

# continuous_update=False: il grafico si aggiorna quando rilasci lo slider
interact(prova_retta,
         w=FloatSlider(min=0, max=80, step=0.5, value=20, description="w", continuous_update=False),
         b=FloatSlider(min=-12000, max=4000, step=50, value=0, description="b", continuous_update=False));

*Se gli slider non compaiono, puoi chiamare la funzione a mano, per esempio `prova_retta(30, -2000)`.*

In [ ]:
# ✏️ scrivi qui la tua retta migliore
mio_w = 20.0
mio_b = 0.0

> 🤔 Perché è così difficile trovarla a mano? Quando cambi $w$ la retta **ruota attorno al punto con pinna = 0**, che è lontanissimo dai dati: ogni piccolo cambio di pendenza la sposta di migliaia di grammi. Ricordatelo per la sezione 2.5.

### 2.4 Ora lo fa l'algoritmo: `LinearRegression`

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

regressione = LinearRegression()             # 1) creo il modello
regressione.fit(X_peso_train, y_peso_train)  # 2) training: trova w e b

w_appreso = regressione.coef_[0]      # coef_ contiene un peso per ogni feature (qui una sola)
b_appreso = regressione.intercept_
print(f"Retta appresa: ŷ = {w_appreso:.1f} · pinna {b_appreso:+.0f}")

Il modello è **interpretabile**: ogni millimetro di pinna in più vale circa `w_appreso` grammi di peso.

Confrontiamo la tua retta con quella appresa, sul training e sul test set. L'MSE è in *grammi al quadrato*, poco intuitivo: per leggerlo meglio calcoliamo anche la sua **radice quadrata**, che è un "errore tipico" in grammi.

In [ ]:
righe = []
for nome, w, b in [("La mia retta", mio_w, mio_b), ("LinearRegression", w_appreso, b_appreso)]:
    mse_train = mse(y_peso_train, w * X_peso_train["pinna"] + b)
    mse_test = mse(y_peso_test, w * X_peso_test["pinna"] + b)
    righe.append({"modello": nome, "MSE train": mse_train, "MSE test": mse_test,
                  "errore tipico test (g)": np.sqrt(mse_test)})   # np.sqrt = radice quadrata

# verifico la mia funzione mse con quella di sklearn
print("Controllo con sklearn:", mean_squared_error(y_peso_test, regressione.predict(X_peso_test)))
pd.DataFrame(righe).round(0)

### 2.5 ⏩ Come ha fatto? La discesa del gradiente
A lezione avete visto l'idea: si parte da valori a caso, si calcola l'MSE e si **modifica leggermente** $w$ e $b$ nella direzione in cui l'errore diminuisce, ripetendo molte volte. La direzione si ricava dalla **derivata** dell'MSE (il *gradiente*).

*Nota onesta: `LinearRegression` in realtà usa una formula diretta (i minimi quadrati), che per la regressione lineare esiste. La discesa del gradiente serve per tutti i modelli in cui la formula diretta non c'è, per esempio le reti neurali che vedrete più avanti. Qui la implementiamo noi per vedere come funziona.*

Per non incappare nel problema della rotazione visto in 2.3, prima **standardizziamo** la pinna: misuriamo ogni valore come distanza dalla media, in "unità di deviazione standard".

In [ ]:
x = X_peso_train["pinna"].to_numpy()     # .to_numpy() converte la colonna in un array NumPy
y = y_peso_train.to_numpy()
x_std = (x - x.mean()) / x.std()         # pinna standardizzata: media 0, deviazione standard 1

w, b = 0.0, 0.0      # 1) valori iniziali
passo = 0.1          # learning rate: quanto è lungo ogni passo
storia = []          # qui salviamo (iterazione, w, b, MSE) per i grafici

for iterazione in range(50):
    pred = w * x_std + b                        # 2) predizioni con i parametri attuali
    errore = pred - y
    storia.append((iterazione, w, b, np.mean(errore ** 2)))   # 3) MSE attuale
    grad_w = 2 * np.mean(errore * x_std)        # derivata dell'MSE rispetto a w
    grad_b = 2 * np.mean(errore)                # derivata dell'MSE rispetto a b
    w = w - passo * grad_w                      # 4) un piccolo passo "in discesa"
    b = b - passo * grad_b
                                                # 5) ...e il ciclo for ripete

# riporto w e b nella scala originale della pinna (in mm), per confrontarli con LinearRegression
w_mm = w / x.std()
b_mm = b - w * x.mean() / x.std()
print(f"Discesa del gradiente: ŷ = {w_mm:.1f} · pinna {b_mm:+.0f}")
print(f"LinearRegression:      ŷ = {w_appreso:.1f} · pinna {b_appreso:+.0f}")

In [ ]:
storia = pd.DataFrame(storia, columns=["iterazione", "w", "b", "MSE"])   # lista di tuple → DataFrame

fig, assi = plt.subplots(1, 2, figsize=(14, 4.5))
assi[0].plot(storia["iterazione"], storia["MSE"], marker=".")
assi[0].set_yscale("log")       # scala logaritmica: l'MSE iniziale è enorme
assi[0].set_xlabel("iterazione"); assi[0].set_ylabel("MSE (scala log)"); assi[0].grid(alpha=0.3)
assi[0].set_title("L'errore scende a ogni passo")

assi[1].scatter(x, y, edgecolors="k", linewidths=0.5, s=25, color="lightgray")
xs = np.array([170, 235])
for it in [0, 1, 3, 6, 49]:
    riga = storia.iloc[it]
    w_it = riga["w"] / x.std()
    b_it = riga["b"] - riga["w"] * x.mean() / x.std()
    assi[1].plot(xs, w_it * xs + b_it, linewidth=2, label=f"iterazione {it}")
assi[1].set_ylim(0, 7000); assi[1].legend(); assi[1].grid(alpha=0.3)
assi[1].set_xlabel("pinna (mm)"); assi[1].set_ylabel("peso (g)")
assi[1].set_title("La retta si avvicina ai dati")
plt.show()

> 🤔 Prova a cambiare `passo` (per esempio 0.01 oppure 1.05) e riesegui le due celle. Cosa succede?

### 2.6 Più misure: dalla retta al piano
Con **due** feature il modello lineare diventa un **piano**:

$$\hat{y} = w_0 + w_{pinna} \cdot x_{pinna} + w_{becco} \cdot x_{becco}$$

Il grafico qui sotto è **interattivo**: trascinalo con il mouse per ruotarlo. Lo disegniamo con **plotly**, una libreria per grafici interattivi.

In [ ]:
import plotly.graph_objects as go

X2 = df[["pinna", "becco_lunghezza"]]
X2_train, X2_test, y2_train, y2_test = train_test_split(X2, y_peso, test_size=0.3, random_state=0)
regressione2 = LinearRegression().fit(X2_train, y2_train)
print("Pesi:", {c: round(float(v), 1) for c, v in zip(X2.columns, regressione2.coef_)}, "| intercetta:", round(regressione2.intercept_))

# il piano: calcolo la predizione su una griglia di valori (pinna, becco)
pp, bb = np.meshgrid(np.linspace(170, 235, 20), np.linspace(32, 60, 20))
griglia = pd.DataFrame({"pinna": pp.ravel(), "becco_lunghezza": bb.ravel()})
zz = regressione2.predict(griglia).reshape(pp.shape)

fig = go.Figure()
fig.add_trace(go.Scatter3d(x=X2_train["pinna"], y=X2_train["becco_lunghezza"], z=y2_train,
                           mode="markers", marker=dict(size=3, color="steelblue"), name="pinguini"))
fig.add_trace(go.Surface(x=pp, y=bb, z=zz, opacity=0.5, showscale=False,
                         colorscale=[[0, "crimson"], [1, "crimson"]], name="piano"))
fig.update_layout(height=550, margin=dict(l=0, r=0, t=30, b=0),
                  scene=dict(xaxis_title="pinna (mm)", yaxis_title="becco_lunghezza (mm)", zaxis_title="peso (g)"))
fig.show()

E se volessimo usare anche **specie**, **isola** e **sesso**? Sono parole, ma il modello lavora solo con numeri: le categorie vanno **codificate**. Il modo più comune è il **one-hot encoding**: una colonna 0/1 per ogni valore possibile.

> 🐍 `pd.get_dummies` crea le colonne 0/1. Con `drop_first=True` toglie la prima di ogni gruppo, perché è ridondante: se un pinguino non è né Chinstrap né Gentoo, è per forza Adelie.

In [ ]:
X_tutte = pd.get_dummies(df.drop(columns=["peso"]),            # tutte le colonne tranne il target
                         columns=["specie", "isola", "sesso"], drop_first=True, dtype=int)
X_tutte.head()

In [ ]:
insiemi_di_feature = {
    "solo pinna": ["pinna"],
    "pinna + becco_lunghezza": ["pinna", "becco_lunghezza"],
    "tutte (con le categorie)": list(X_tutte.columns),
}

righe = []
for nome, colonne in insiemi_di_feature.items():
    Xa, Xb, ya, yb = train_test_split(X_tutte[colonne], y_peso, test_size=0.3, random_state=0)  # stessa divisione di prima
    modello = LinearRegression().fit(Xa, ya)
    righe.append({"feature": nome, "errore tipico test (g)": np.sqrt(mse(yb, modello.predict(Xb)))})
pd.DataFrame(righe).round(0)

> 🤔 Aggiungere il becco non serve quasi a niente, aggiungere specie e sesso invece migliora molto. Perché? Guarda i pesi del modello completo: quanto pesa in più, a parità di misure, un maschio? E un Gentoo?

<details>
<summary><b>💡 Per vedere i pesi</b></summary>

```python
modello_completo = LinearRegression().fit(X_tutte, y_peso)
pd.Series(modello_completo.coef_, index=X_tutte.columns).round(1)
```

</details>

---
# Parte 3 — *"Riconoscete la specie"*

> 📩 *Ottimo! Adesso un'altra richiesta: da una foto del becco riusciamo a misurarne lunghezza e profondità. Potete dirci di che specie è il pinguino? Ci interessano soprattutto Adelie e Chinstrap, che da lontano si confondono.*

### 3.1 Classificare con la regressione
A lezione avete visto un trucco: **trasformare le classi in numeri** (Chinstrap = **+1**, Adelie = **−1**), addestrare una regressione e poi classificare con una **soglia**: se $\hat y \geq 0$ dico Chinstrap, altrimenti Adelie.

In [ ]:
due_specie = df[df["specie"].isin(["Adelie", "Chinstrap"])]     # .isin: tiene solo le righe con quei valori
X_due = due_specie[["becco_lunghezza", "becco_profondita"]]
y_due = due_specie["specie"]

X_due_train, X_due_test, y_due_train, y_due_test = train_test_split(
    X_due, y_due, test_size=0.3, random_state=0, stratify=y_due)

# np.where(condizione, valore_se_vero, valore_se_falso) lavora elemento per elemento
y_numerico = np.where(y_due_train == "Chinstrap", 1, -1)

regressione_specie = LinearRegression().fit(X_due_train, y_numerico)

# cosa restituisce la regressione? numeri "continui", non classi!
uscite = regressione_specie.predict(X_due_test.head(8))
pd.DataFrame({"uscita della regressione": uscite.round(2),
              "classe con soglia 0": np.where(uscite >= 0, "Chinstrap", "Adelie"),
              "classe vera": y_due_test.head(8).to_numpy()})

In [ ]:
from sklearn.metrics import accuracy_score

def classificatore_lineare(X):
    """Regressione + soglia: restituisce un'etichetta per ogni riga di X."""
    return np.where(regressione_specie.predict(X) >= 0, "Chinstrap", "Adelie")

print("Accuracy sul test set:", accuracy_score(y_due_test, classificatore_lineare(X_due_test)))

### 3.2 La frontiera decisionale
Il modello lineare calcola un **piano** sopra il grafico (a sinistra, il colore è il valore di $\hat y$). La soglia $\hat y = 0$ taglia il piano lungo una **retta**: quella retta è la **frontiera decisionale** (a destra).

In [ ]:
fig, assi = plt.subplots(1, 2, figsize=(15, 5))

# a sinistra: il valore continuo della regressione su una griglia di punti
ll, pp = np.meshgrid(np.linspace(31, 59, 200), np.linspace(15, 22, 200))
griglia = pd.DataFrame({"becco_lunghezza": ll.ravel(), "becco_profondita": pp.ravel()})
valori = regressione_specie.predict(griglia).reshape(ll.shape)
mappa = assi[0].contourf(ll, pp, valori, levels=20, cmap="PuOr", vmin=-2.5, vmax=2.5)
assi[0].contour(ll, pp, valori, levels=[0], colors="k", linewidths=2)    # la linea dove ŷ = 0
fig.colorbar(mappa, ax=assi[0], label="ŷ (uscita della regressione)")
assi[0].scatter(X_due_train["becco_lunghezza"], X_due_train["becco_profondita"],
                c=[COLORI[s] for s in y_due_train], edgecolors="k", linewidths=0.5, s=30)
assi[0].set_xlabel("becco_lunghezza"); assi[0].set_ylabel("becco_profondita")
assi[0].set_title("Il piano della regressione (linea nera: ŷ = 0)")

# a destra: le due regioni di decisione
disegna_frontiera(classificatore_lineare, X_due_train, y_due_train, "Frontiera decisionale: una retta", ax=assi[1])
plt.show()

📌 Con un modello lineare la frontiera è sempre una **retta** (con 3 feature un piano, con più feature un iperpiano). Qui è **obliqua**: dipende da entrambe le misure insieme.

### 3.3 E con un albero?
Facciamo lo stesso con l'albero di decisione dell'Esercitazione 1.

In [ ]:
from sklearn.tree import DecisionTreeClassifier

albero_due = DecisionTreeClassifier(max_depth=3, random_state=0).fit(X_due_train, y_due_train)
print("Accuracy test – lineare:", accuracy_score(y_due_test, classificatore_lineare(X_due_test)))
print("Accuracy test – albero: ", accuracy_score(y_due_test, albero_due.predict(X_due_test)))

fig, assi = plt.subplots(1, 2, figsize=(15, 5))
disegna_frontiera(classificatore_lineare, X_due_train, y_due_train, "Regressione + soglia", ax=assi[0])
disegna_frontiera(albero_due, X_due_train, y_due_train, "Albero, max_depth=3", ax=assi[1])
plt.show()

> 🤔 La frontiera dell'albero è fatta **solo di segmenti orizzontali e verticali**. Perché? *(Pensa a com'è fatta ogni domanda dell'albero.)*

### 3.4 Tutte e tre le specie
Con tre classi il trucco della soglia non basta più (servirebbe un modello per classe, o la *regressione logistica* che vedrete più avanti). L'albero invece gestisce **direttamente** il caso multiclasse.

In [ ]:
X = df[["becco_lunghezza", "becco_profondita"]]
y = df["specie"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)

albero_specie = DecisionTreeClassifier(max_depth=3, random_state=0).fit(X_train, y_train)
disegna_frontiera(albero_specie, X_train, y_train, "Tre specie, albero con max_depth=3")

---
# Parte 4 — *"Come decide il vostro albero? Possiamo fidarci?"*

> 📩 *Funziona! Prima di usarlo sul campo, però, vorremmo capire come ragiona. E come facciamo a sapere che non ci sta prendendo in giro?*

Quando chiamiamo `.fit()` su un albero, l'algoritmo fa questo:
1. parte dalla **radice** con tutti i dati;
2. prova **ogni feature** e **ogni soglia** possibile;
3. sceglie la domanda che riduce di più l'**impurità**;
4. divide i dati e **ripete** su ciascun figlio;
5. si ferma quando un nodo è puro o quando scatta un **criterio di stop**.

È un algoritmo **greedy**: a ogni nodo sceglie la domanda migliore *in quel momento*, senza tornare indietro.

### 4.1 Misurare l'impurità: Gini ed entropia 
Per un nodo con classi in proporzioni $p_k$:

$$G = 1 - \sum_k p_k^2 \qquad\qquad H = -\sum_k p_k \log_2 p_k$$

Entrambe valgono **0 quando il nodo è puro** e sono massime quando le classi sono mescolate in parti uguali.

In [ ]:
p = np.linspace(0, 1, 201)       # 201 valori equidistanti tra 0 e 1: la proporzione della classe 1

gini_curva = 1 - p ** 2 - (1 - p) ** 2
with np.errstate(divide="ignore", invalid="ignore"):          # log2(0) darebbe un avviso: lo ignoriamo
    entropia_curva = -(p * np.log2(p) + (1 - p) * np.log2(1 - p))
entropia_curva = np.nan_to_num(entropia_curva)                 # e mettiamo 0 al posto dei NaN agli estremi

plt.figure(figsize=(7, 4.5))
plt.plot(p, gini_curva, label="Gini", linewidth=2)
plt.plot(p, entropia_curva, label="Entropia", linewidth=2)
plt.xlabel("p = proporzione della classe 1"); plt.ylabel("impurità")
plt.title("Impurità di un nodo con due classi"); plt.legend(); plt.grid(alpha=0.3)
plt.show()

### 4.2 ✏️ Esercizio: la funzione `gini`
Completa la funzione. Le proporzioni $p_k$ le calcola già `value_counts(normalize=True)`; prova a eseguirla da sola per vedere cosa restituisce:

In [ ]:
y_train.value_counts(normalize=True)     # normalize=True: proporzioni invece di conteggi

In [ ]:
def gini(etichette):
    """Indice di Gini di un gruppo di esempi (etichette: lista, array o colonna pandas)."""
    proporzioni = pd.Series(etichette).value_counts(normalize=True)   # una p_k per ogni classe
    # ✏️ restituisci 1 meno la somma dei quadrati delle proporzioni
    return 0

print(gini(["A", "A", "A", "A"]))                   # nodo puro: 0.0
print(gini(["A", "B", "A", "B"]))                   # metà e metà: 0.5
print(round(gini(["Sì"] * 5 + ["No"] * 3), 3))      # 0.469: la radice di Play Tennis vista nella lezione 7!
print(round(gini(y_train), 3))                      # la radice del nostro albero delle specie

<details>
<summary><b>💡 Soluzione</b> (clicca per aprire, ma prima prova da solo!)</summary>

```python
def gini(etichette):
    proporzioni = pd.Series(etichette).value_counts(normalize=True)
    return 1 - (proporzioni ** 2).sum()

print(gini(["A", "A", "A", "A"]))
print(gini(["A", "B", "A", "B"]))
print(round(gini(["Sì"] * 5 + ["No"] * 3), 3))
print(round(gini(y_train), 3))
```

</details>

L'entropia si scrive allo stesso modo:

In [ ]:
def entropia(etichette):
    proporzioni = pd.Series(etichette).value_counts(normalize=True)
    return -(proporzioni * np.log2(proporzioni)).sum()

print(round(entropia(["Sì"] * 5 + ["No"] * 3), 3))   # 0.954 bit

### 4.3 Rifacciamo noi il primo passo del `.fit()`
Con le **feature numeriche** l'albero cerca una **soglia**:
1. ordina i valori della feature;
2. prende come **candidati** i punti medi tra valori consecutivi;
3. per ogni soglia divide i dati in *sinistra* ($x \leq$ soglia) e *destra* ($x >$ soglia) e calcola l'impurità dei due figli, **pesata** per il numero di esempi;
4. sceglie la soglia con l'impurità pesata **minima**.

*(L'algoritmo vero considera solo i punti medi in cui la classe cambia, perché gli altri non possono vincere: per semplicità noi li proviamo tutti.)*

In [ ]:
def impurita_split(x, y, soglia, misura=gini):
    """Impurità pesata dei due figli ottenuti con la domanda "x <= soglia?"."""
    sinistra = y[x <= soglia]
    destra = y[x > soglia]
    n = len(y)
    return len(sinistra) / n * misura(sinistra) + len(destra) / n * misura(destra)

def soglie_candidate(x):
    valori = np.unique(x)                     # valori distinti, già ordinati
    return (valori[:-1] + valori[1:]) / 2     # punti medi tra valori consecutivi

y_np = y_train.to_numpy()
fig, assi = plt.subplots(1, 2, figsize=(15, 4.5))
for ax, feature in zip(assi, X_train.columns):
    x_np = X_train[feature].to_numpy()
    soglie = soglie_candidate(x_np)
    impurita = [impurita_split(x_np, y_np, s) for s in soglie]     # una impurità per ogni soglia
    migliore = soglie[np.argmin(impurita)]                         # np.argmin: posizione del minimo
    ax.plot(soglie, impurita, marker=".", markersize=3)
    ax.axvline(migliore, color="crimson", linestyle="--", label=f"migliore: {migliore:.2f}")
    ax.axhline(gini(y_np), color="gray", linestyle=":", label=f"Gini della radice: {gini(y_np):.3f}")
    ax.set_xlabel(f"soglia su {feature}"); ax.set_ylabel("Gini pesato dei figli")
    ax.set_title(f"{feature}: Gini minimo = {min(impurita):.3f}"); ax.legend(); ax.grid(alpha=0.3)
plt.show()

Vince la feature con il Gini pesato più basso, cioè con la **riduzione di impurità** più grande. Controlliamo che scikit-learn abbia fatto la stessa scelta:

In [ ]:
from sklearn.tree import plot_tree

radice = DecisionTreeClassifier(max_depth=1, random_state=0).fit(X_train, y_train)

feature_scelta = X_train.columns[radice.tree_.feature[0]]    # tree_ contiene la struttura interna dell'albero
soglia_scelta = radice.tree_.threshold[0]
print(f"sklearn: {feature_scelta} <= {soglia_scelta:.2f}")

plt.figure(figsize=(9, 4))
# attenzione: plot_tree usa colori suoi, diversi da quelli dei nostri grafici
plot_tree(radice, feature_names=list(X_train.columns), class_names=list(radice.classes_),
          filled=True, rounded=True)
plt.show()

**Information gain.** Con l'entropia il ragionamento è identico, ma si guarda il **guadagno**: entropia del padre meno entropia pesata dei figli.

$$IG = H(\text{padre}) - \sum_i \frac{n_i}{n} H(\text{figlio}_i)$$

In [ ]:
x_np = X_train["becco_lunghezza"].to_numpy()
soglie = soglie_candidate(x_np)
guadagni = [entropia(y_np) - impurita_split(x_np, y_np, s, misura=entropia) for s in soglie]
migliore = soglie[np.argmax(guadagni)]          # qui cerchiamo il MASSIMO

radice_entropia = DecisionTreeClassifier(max_depth=1, criterion="entropy", random_state=0).fit(X_train, y_train)
print(f"La nostra soglia: {migliore:.2f}  |  sklearn con criterion='entropy': {radice_entropia.tree_.threshold[0]:.2f}")

plt.figure(figsize=(7.5, 4.5))
plt.plot(soglie, guadagni, marker=".", markersize=3, color="tab:orange")
plt.axvline(migliore, color="crimson", linestyle="--")
plt.xlabel("soglia su becco_lunghezza"); plt.ylabel("information gain (bit)")
plt.title(f"Information gain massimo = {max(guadagni):.3f} bit"); plt.grid(alpha=0.3)
plt.show()

📌 Gini ed entropia di solito scelgono soglie molto simili. scikit-learn implementa l'algoritmo **CART** : split sempre **binari**, criterio Gini di default oppure entropia con `criterion="entropy"`, e la possibilità di fare **regressione**. ID3 e C4.5 usano invece l'information gain (o il *gain ratio*) e split con più rami.

### 4.4 Alberi di regressione 
Torniamo al peso. Un albero può fare anche **regressione**: ogni foglia predice la **media** dei pesi dei pinguini che ci arrivano, e l'impurità di un nodo è la sua **varianza**, cioè l'MSE rispetto alla media.

In [ ]:
from sklearn.tree import DecisionTreeRegressor

pinne = pd.DataFrame({"pinna": np.linspace(150, 260, 500)})   # anche FUORI dal range dei dati (172-231 mm)

plt.figure(figsize=(10, 5.5))
plt.scatter(X_peso_train["pinna"], y_peso_train, color="lightgray", edgecolors="k", linewidths=0.4, s=25)
plt.axvspan(150, X_peso["pinna"].min(), color="gold", alpha=0.15)     # zone senza dati di training
plt.axvspan(X_peso["pinna"].max(), 260, color="gold", alpha=0.15, label="fuori dal range dei dati")
plt.plot(pinne["pinna"], regressione.predict(pinne), color="k", linestyle="--", label="regressione lineare")

for profondita in [1, 2, 4]:
    albero_reg = DecisionTreeRegressor(max_depth=profondita, random_state=0).fit(X_peso_train, y_peso_train)
    errore = np.sqrt(mse(y_peso_test, albero_reg.predict(X_peso_test)))
    plt.plot(pinne["pinna"], albero_reg.predict(pinne), linewidth=2,
             label=f"albero max_depth={profondita} (errore tipico test {errore:.0f} g)")

plt.xlabel("pinna (mm)"); plt.ylabel("peso (g)"); plt.legend(fontsize=9); plt.grid(alpha=0.3)
plt.title("Alberi di regressione: funzioni a gradini")
plt.show()

In [ ]:
# Arrivano due pinguini fuori dal comune: pinna di 160 mm e di 250 mm. Quanto pesano?
nuovi = pd.DataFrame({"pinna": [160, 250]})
albero_reg = DecisionTreeRegressor(max_depth=4, random_state=0).fit(X_peso_train, y_peso_train)
pd.DataFrame({"pinna (mm)": nuovi["pinna"],
              "regressione lineare (g)": regressione.predict(nuovi).round(0),
              "albero max_depth=4 (g)": albero_reg.predict(nuovi).round(0)})

📌 L'albero predice una **funzione costante a tratti** (a gradini). Fuori dal range dei dati di training **non estrapola**: resta fermo sull'ultimo gradino . La retta invece continua a crescere. Chi ha ragione dipende dal problema, ma è un limite da conoscere, soprattutto con trend e serie temporali.

### 4.5 Una richiesta più difficile: il sesso
> 📩 *Vi ricordate i 9 pinguini senza sesso? Potete stimarlo dalle misure del becco?*

In [ ]:
X_sesso = df[["becco_lunghezza", "becco_profondita"]]
y_sesso = df["sesso"]
X_sesso_train, X_sesso_test, y_sesso_train, y_sesso_test = train_test_split(
    X_sesso, y_sesso, test_size=0.3, random_state=0, stratify=y_sesso)

mostra_punti(X_sesso_train, y_sesso_train, "Maschi e femmine: molto più mescolati delle specie")

🔒 Questa volta il **test set lo chiudiamo a chiave** fino alla fine: lo useremo **una sola volta**, per il verdetto finale. Lasciamo crescere un albero senza limiti e guardiamo solo il training set:

In [ ]:
albero_libero = DecisionTreeClassifier(random_state=0).fit(X_sesso_train, y_sesso_train)
print("Profondità:", albero_libero.get_depth(), "| foglie:", albero_libero.get_n_leaves())
print("Accuracy sul TRAINING set:", accuracy_score(y_sesso_train, albero_libero.predict(X_sesso_train)))
disegna_frontiera(albero_libero, X_sesso_train, y_sesso_train, "Albero senza limiti")

> 🤔 100% sul training, con tante "isolette" create per singoli pinguini. Ti ricorda il **memorizzatore** dell'Esercitazione 1?

### 4.6 Quando fermarsi: iperparametri e pruning 
Senza limiti l'albero si ferma solo quando ogni foglia è pura: con dati mescolati vuol dire **una foglia per quasi ogni esempio**. Per evitarlo ci sono due strade:
- **pre-pruning**: si ferma la crescita in anticipo con gli iperparametri `max_depth`, `min_samples_split`, `min_samples_leaf`, `max_leaf_nodes`, `min_impurity_decrease`;
- **post-pruning**: si fa crescere l'albero completo e poi si tagliano i rami poco utili. In scikit-learn c'è il *cost-complexity pruning*, regolato da `ccp_alpha`.

Vediamo l'effetto di ciascuno sulla frontiera:

> 🐍 `DecisionTreeClassifier(**parametri)`: i due asterischi "spacchettano" un dizionario in argomenti. Se `parametri = {"max_depth": 3}`, la chiamata equivale a `DecisionTreeClassifier(max_depth=3)`.

In [ ]:
configurazioni = {
    "senza limiti": {},
    "max_depth=3": {"max_depth": 3},
    "min_samples_leaf=10": {"min_samples_leaf": 10},
    "max_leaf_nodes=6": {"max_leaf_nodes": 6},
    "min_impurity_decrease=0.01": {"min_impurity_decrease": 0.01},
    "ccp_alpha=0.01 (post-pruning)": {"ccp_alpha": 0.01},
}

fig, assi = plt.subplots(2, 3, figsize=(18, 10))
for ax, (nome, parametri) in zip(assi.ravel(), configurazioni.items()):    # .ravel(): da matrice 2x3 a lista di 6
    modello = DecisionTreeClassifier(random_state=0, **parametri).fit(X_sesso_train, y_sesso_train)
    acc = accuracy_score(y_sesso_train, modello.predict(X_sesso_train))
    disegna_frontiera(modello, X_sesso_train, y_sesso_train,
                      f"{nome}\n{modello.get_n_leaves()} foglie, accuracy training {acc:.2f}", ax=ax)
plt.tight_layout()
plt.show()

Il **post-pruning** penalizza gli alberi con tante foglie: minimizza $R(T) + \alpha \cdot |\text{foglie}|$, dove $R(T)$ è l'errore sul training . Più $\alpha$ è grande, più rami vengono tagliati. scikit-learn calcola per noi i valori di $\alpha$ in cui l'albero cambia:

In [ ]:
percorso = DecisionTreeClassifier(random_state=0).cost_complexity_pruning_path(X_sesso_train, y_sesso_train)
alpha = percorso.ccp_alphas[:-1]       # l'ultimo valore taglia tutto e lascia solo la radice: lo escludiamo

foglie = [DecisionTreeClassifier(random_state=0, ccp_alpha=a).fit(X_sesso_train, y_sesso_train).get_n_leaves()
          for a in alpha]

plt.figure(figsize=(7.5, 4.5))
plt.step(alpha[1:], foglie[1:], where="post")      # il primo alpha vale 0: lo saltiamo per usare la scala log
plt.xscale("log")
plt.xlabel("ccp_alpha (scala log)"); plt.ylabel("numero di foglie"); plt.grid(alpha=0.3)
plt.title("Più alpha è grande, più l'albero viene potato")
plt.show()

### 4.7 ✏️ Quale albero scegliamo? La cross-validation
Abbiamo tanti alberi possibili: quale consegniamo? **Non possiamo sceglierlo guardando il test set**: sarebbe l'*apprendimento occulto dal test set*. Ci serve una stima della generalizzazione che usi **solo il training set**.

La soluzione è la **k-fold cross-validation**: si divide il training set in $k$ parti (*fold*). A turno, una fa da **validation set** e le altre servono per addestrare. Si ottengono $k$ punteggi, se ne fa la media.

In [ ]:
k = 5
schema = np.zeros((k, k))
np.fill_diagonal(schema, 1)       # 1 sulla diagonale: il fold usato per la validazione

plt.figure(figsize=(9, 3.5))
plt.imshow(schema, aspect="auto", cmap=ListedColormap(["#A8C8E8", "#F5A623"]))
for i in range(k):
    for j in range(k):
        plt.text(j, i, "validation" if i == j else "train", ha="center", va="center", fontsize=8)
plt.xticks(range(k), [f"fold {j+1}" for j in range(k)])
plt.yticks(range(k), [f"giro {i+1}" for i in range(k)])
plt.title("5-fold cross-validation (tutto dentro il training set)")
plt.show()

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

# StratifiedKFold: k-fold che mantiene in ogni fold la stessa proporzione di maschi e femmine (L4)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)

# cross_val_score fa da solo i 5 addestramenti e le 5 valutazioni
punteggi = cross_val_score(DecisionTreeClassifier(max_depth=3, random_state=0),
                           X_sesso_train, y_sesso_train, cv=cv)
print("Accuracy nei 5 fold:", punteggi.round(3))
print(f"Media: {punteggi.mean():.3f}  ± {punteggi.std():.3f} (deviazione standard)")

Ora tocca a te: per ogni profondità da 1 a 12 calcola l'**accuracy sul training set** e l'**accuracy media in cross-validation** (con la sua deviazione standard). Il grafico in fondo alla cella è già pronto.

In [ ]:
profondita = list(range(1, 13))
acc_train, acc_cv_media, acc_cv_std = [], [], []

for d in profondita:
    modello = DecisionTreeClassifier(max_depth=d, random_state=0)
    # ✏️ 1) addestra modello su tutto il training set e calcola l'accuracy sul training set
    # ✏️ 2) calcola i punteggi di cross-validation con cross_val_score (usa cv=cv)
    # ✏️ 3) aggiungi i tre risultati alle liste acc_train, acc_cv_media, acc_cv_std con .append()
    pass

n = len(acc_train)
plt.figure(figsize=(8.5, 5))
plt.plot(profondita[:n], acc_train, marker="o", label="training set")
plt.errorbar(profondita[:n], acc_cv_media, yerr=acc_cv_std, marker="s", capsize=4, label="cross-validation")
plt.xlabel("max_depth"); plt.ylabel("accuracy"); plt.xticks(profondita)
plt.legend(); plt.grid(alpha=0.4); plt.title("Training contro validazione")
plt.show()

<details>
<summary><b>💡 Soluzione</b> (clicca per aprire, ma prima prova da solo!)</summary>

```python
profondita = list(range(1, 13))
acc_train, acc_cv_media, acc_cv_std = [], [], []

for d in profondita:
    modello = DecisionTreeClassifier(max_depth=d, random_state=0)
    modello.fit(X_sesso_train, y_sesso_train)
    acc_train.append(accuracy_score(y_sesso_train, modello.predict(X_sesso_train)))
    punteggi = cross_val_score(modello, X_sesso_train, y_sesso_train, cv=cv)
    acc_cv_media.append(punteggi.mean())
    acc_cv_std.append(punteggi.std())

n = len(acc_train)
plt.figure(figsize=(8.5, 5))
plt.plot(profondita[:n], acc_train, marker="o", label="training set")
plt.errorbar(profondita[:n], acc_cv_media, yerr=acc_cv_std, marker="s", capsize=4, label="cross-validation")
plt.xlabel("max_depth"); plt.ylabel("accuracy"); plt.xticks(profondita)
plt.legend(); plt.grid(alpha=0.4); plt.title("Training contro validazione")
plt.show()
```

</details>

### 📌 Come si legge il grafico
È il grafico della slide *Overfitting* della lezione 8, con i nostri dati:
- a **sinistra** l'albero è troppo semplice e sbaglia anche sul training: **underfitting**;
- a **destra** il training arriva al 100% mentre la validazione **scende**: **overfitting**;
- la profondità da scegliere è quella con la **validazione migliore**. Le barre ricordano che la stima è "rumorosa": tra due profondità con punteggi entro le barre, conviene la più semplice.

scikit-learn fa tutto questo, anche su più iperparametri insieme, con **`GridSearchCV`**: prova ogni combinazione con la cross-validation e tiene la migliore.

In [ ]:
from sklearn.model_selection import GridSearchCV

griglia_parametri = {"max_depth": list(range(1, 11)),          # dizionario: iperparametro → valori da provare
                     "min_samples_leaf": [1, 2, 5, 10, 20]}
ricerca = GridSearchCV(DecisionTreeClassifier(random_state=0), griglia_parametri, cv=cv)
ricerca.fit(X_sesso_train, y_sesso_train)      # 10 x 5 combinazioni x 5 fold = 250 addestramenti!

print("Iperparametri migliori:", ricerca.best_params_)
print(f"Accuracy in cross-validation: {ricerca.best_score_:.3f}")
albero_sesso = ricerca.best_estimator_         # l'albero migliore, già riaddestrato su tutto il training set

🔓 **Il verdetto finale.** Abbiamo scelto senza mai guardare il test set: ora possiamo aprirlo, **una volta sola**.

In [ ]:
print("Test – albero senza limiti:", accuracy_score(y_sesso_test, albero_libero.predict(X_sesso_test)))
print("Test – albero scelto con CV:", accuracy_score(y_sesso_test, albero_sesso.predict(X_sesso_test)))
disegna_frontiera(albero_sesso, X_sesso_train, y_sesso_train, f"Albero scelto: {ricerca.best_params_}")

E finalmente rispondiamo alla ricercatrice: ecco il sesso stimato per i 9 pinguini misteriosi.

In [ ]:
risposta = misteriosi[["specie", "becco_lunghezza", "becco_profondita"]].copy()
risposta["sesso stimato"] = albero_sesso.predict(misteriosi[["becco_lunghezza", "becco_profondita"]])
risposta

> 🤔 **Sul tipo di split.** Qui abbiamo diviso i dati a caso (stratificando). Quando non andrebbe bene? Pensa a due casi: se le misure fossero state raccolte in anni diversi e volessimo prevedere quelle dell'anno prossimo (→ **split temporale**); se avessimo più misure dello **stesso pinguino** in anni diversi (→ **split per gruppi**, tutte le misure di un pinguino nello stesso insieme).

### 4.8 Il verdetto multiclasse: le specie
Torniamo alle tre specie e valutiamo l'albero (scelto anche qui con la cross-validation) sul test set.

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, classification_report, precision_score

ricerca_specie = GridSearchCV(DecisionTreeClassifier(random_state=0), {"max_depth": list(range(1, 9))}, cv=cv)
ricerca_specie.fit(X_train, y_train)
albero_specie = ricerca_specie.best_estimator_
pred_specie = albero_specie.predict(X_test)
print("max_depth scelto:", ricerca_specie.best_params_["max_depth"])

ConfusionMatrixDisplay.from_predictions(y_test, pred_specie, cmap="Blues")
plt.title("Matrice di confusione 3x3 (righe = classe vera, colonne = predetta)")
plt.show()

Con più classi ogni classe ha la sua **precision** e la sua **recall**: la classe in esame è "la positiva", tutte le altre insieme sono "la negativa". `classification_report` le calcola tutte:

In [ ]:
print(classification_report(y_test, pred_specie, digits=3))

Per riassumere in un solo numero ci sono due modi:
- **macro average**: calcolo la metrica per ogni classe e faccio la media. **Ogni classe pesa uguale**, anche quella con pochi esempi;
- **micro average**: metto insieme TP e FP di tutte le classi e calcolo la metrica una volta sola. **Ogni esempio pesa uguale**, quindi le classi numerose contano di più. *(Nella classificazione multiclasse la micro precision coincide con l'accuracy.)*

In [ ]:
print("precision macro:", round(precision_score(y_test, pred_specie, average="macro"), 3))
print("precision micro:", round(precision_score(y_test, pred_specie, average="micro"), 3))
print("accuracy:       ", round(accuracy_score(y_test, pred_specie), 3))

> 🤔 Quale specie è la più difficile da riconoscere? Perché? *(Guarda la matrice di confusione, quante Chinstrap ci sono nel dataset, e dove stanno nel grafico.)*

### 4.9 Quanto è sicuro?
Un albero di classificazione dà anche una **probabilità**: è la **proporzione delle classi nella foglia** in cui finisce l'esempio. Se in una foglia ci sono 18 Adelie e 2 Chinstrap, l'albero dirà *"Adelie al 90%"*.

Usiamo un albero di profondità 3: tra poco vedremo perché non quello appena scelto.

In [ ]:
albero_semplice = DecisionTreeClassifier(max_depth=3, random_state=0).fit(X_train, y_train)

tabella = pd.DataFrame(albero_semplice.predict_proba(X_test), columns=albero_semplice.classes_, index=X_test.index)
tabella["predetta"] = albero_semplice.predict(X_test)
tabella["vera"] = y_test
tabella["sicurezza"] = tabella[list(albero_semplice.classes_)].max(axis=1)   # la probabilità più alta
tabella.sort_values("sicurezza").head(6).round(2)      # i 6 pinguini su cui l'albero è meno sicuro

In [ ]:
fig, assi = plt.subplots(1, 2, figsize=(15, 5))
disegna_confidenza(albero_semplice, X_train, y_train, "max_depth=3", ax=assi[0])
disegna_confidenza(albero_specie, X_train, y_train,
                   f"L'albero scelto in 4.8 (profondità {albero_specie.get_depth()})", ax=assi[1])
plt.show()

> 🤔 Colore intenso = sicuro, colore sbiadito = incerto. Dove è incerto l'albero di profondità 3? Corrisponde alle zone in cui le specie si mescolano?
>
> E l'albero profondo, perché è **sicuro al 100% quasi ovunque**, anche dove sbaglia? *(Suggerimento: com'è fatta una foglia di un albero molto profondo?)*

### 4.10 ⏩ Il limite dei confini obliqui 
Nella Parte 3 la frontiera tra Adelie e Chinstrap era **obliqua**. Un albero, che fa una domanda su una feature alla volta, la può solo approssimare a **gradini**: più gradini vuol dire più profondità e più rischio di overfitting.

In [ ]:
fig, assi = plt.subplots(1, 3, figsize=(18, 5))
for ax, d in zip(assi, [1, 3, 8]):
    modello = DecisionTreeClassifier(max_depth=d, random_state=0).fit(X_due_train, y_due_train)
    acc = accuracy_score(y_due_test, modello.predict(X_due_test))
    disegna_frontiera(modello, X_due_train, y_due_train,
                      f"max_depth={d}: {modello.get_n_leaves()} foglie, accuracy test {acc:.2f}", ax=ax)
plt.show()

Uno dei rimedi è **creare una feature combinata**. Il rapporto tra lunghezza e profondità del becco "guarda" le due misure insieme, proprio come la retta obliqua.

In [ ]:
X_rapporto_train = pd.DataFrame({"rapporto": X_due_train["becco_lunghezza"] / X_due_train["becco_profondita"]})
X_rapporto_test = pd.DataFrame({"rapporto": X_due_test["becco_lunghezza"] / X_due_test["becco_profondita"]})

ceppo = DecisionTreeClassifier(max_depth=1, random_state=0).fit(X_rapporto_train, y_due_train)   # una sola domanda!
print(f"Una sola domanda: rapporto <= {ceppo.tree_.threshold[0]:.2f}")
print("Accuracy test:", round(accuracy_score(y_due_test, ceppo.predict(X_rapporto_test)), 2),
      "(la regressione + soglia della Parte 3 faceva", round(accuracy_score(y_due_test, classificatore_lineare(X_due_test)), 2), ")")

plt.figure(figsize=(7.5, 4))
for specie in ["Adelie", "Chinstrap"]:
    plt.hist(X_rapporto_train["rapporto"][y_due_train == specie], bins=20, alpha=0.6,
             color=COLORI[specie], label=specie)
plt.axvline(ceppo.tree_.threshold[0], color="k", linestyle="--", label="soglia dell'albero")
plt.xlabel("becco_lunghezza / becco_profondita"); plt.legend(); plt.grid(alpha=0.3)
plt.show()

---
# Parte 5 — *"Si può fare di meglio?"*

> 📩 *Il sesso stimato è giusto circa 8 volte su 10. Si può fare di meglio?*

La lezione 8 si chiudeva con i **limiti** degli alberi:
- un albero **profondo** ha **basso bias** (si adatta bene) ma **alta varianza** (cambia molto da un campione all'altro);
- un albero **poco profondo** ha **alto bias** ma **bassa varianza**.

### 5.1 Vediamo la varianza
Creiamo 4 **campioni bootstrap** del training set: estrazioni **con reinserimento** della stessa dimensione dell'originale (alcuni pinguini compaiono più volte, altri mai). Su ognuno addestriamo un albero profondo.

> 🐍 `rng = np.random.default_rng(0)` crea un generatore di numeri casuali con seme fisso. `rng.choice(n, size=n, replace=True)` estrae `n` numeri tra 0 e `n-1` **con reinserimento**. `.iloc[indici]` prende le righe in quelle posizioni.

In [ ]:
rng = np.random.default_rng(0)
n = len(X_sesso_train)

fig, assi = plt.subplots(2, 2, figsize=(14, 10))
for i, ax in enumerate(assi.ravel()):           # enumerate dà anche il numero del giro: 0, 1, 2, 3
    indici = rng.choice(n, size=n, replace=True)
    X_boot, y_boot = X_sesso_train.iloc[indici], y_sesso_train.iloc[indici]
    albero_boot = DecisionTreeClassifier(random_state=0).fit(X_boot, y_boot)
    disegna_frontiera(albero_boot, X_sesso_train, y_sesso_train, f"Albero {i + 1} (campione bootstrap {i + 1})", ax=ax)
plt.tight_layout()
plt.show()

Stesso algoritmo, dati quasi uguali, frontiere molto diverse: **questa è l'alta varianza**. L'idea degli **ensemble** è la *saggezza della folla*: se tanti modelli sbagliano in modi diversi, **mediando** le loro risposte gli errori si compensano.

### 5.2 Il bootstrap da vicino
Quanti pinguini **diversi** finiscono in un campione bootstrap? Verifichiamolo ripetendo l'estrazione 1000 volte.

In [ ]:
frazioni = [len(np.unique(rng.choice(n, size=n, replace=True))) / n for _ in range(1000)]   # _ = variabile che non usiamo

print(f"Esempi unici in media: {np.mean(frazioni):.1%}  (teoria: 1 - 1/e = {1 - 1/np.e:.1%})")
plt.figure(figsize=(7, 4))
plt.hist(frazioni, bins=20, color="steelblue", edgecolor="k")
plt.axvline(1 - 1 / np.e, color="crimson", linestyle="--", label="1 − 1/e")
plt.xlabel("frazione di esempi unici nel campione bootstrap"); plt.legend(); plt.grid(alpha=0.3)
plt.show()

In ogni campione circa il **63%** dei pinguini è presente e il **37%** resta fuori: sono gli esempi **out-of-bag** (OOB). Ogni albero non li ha mai visti, quindi possiamo usarli per stimare la generalizzazione **senza toccare il test set**.

### 5.3 Random forest
Una **random forest** è il bagging di alberi profondi con un ingrediente in più: a ogni nodo l'albero può scegliere solo tra **alcune feature estratte a caso** (`max_features`, di default $\sqrt{d}$). Così gli alberi si somigliano meno, sbagliano in modi più diversi, e la media funziona meglio.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

foresta = RandomForestClassifier(n_estimators=200, oob_score=True, random_state=0)   # 200 alberi
foresta.fit(X_sesso_train, y_sesso_train)
print(f"Random forest – stima out-of-bag:   {foresta.oob_score_:.3f}")
print(f"Random forest – cross-validation:   {cross_val_score(foresta, X_sesso_train, y_sesso_train, cv=cv).mean():.3f}")
print(f"Un albero profondo – CV:            {cross_val_score(albero_libero, X_sesso_train, y_sesso_train, cv=cv).mean():.3f}")
print(f"Albero scelto con GridSearchCV – CV: {ricerca.best_score_:.3f}")

fig, assi = plt.subplots(1, 2, figsize=(15, 5))
disegna_frontiera(albero_libero, X_sesso_train, y_sesso_train, "Un albero profondo", ax=assi[0])
disegna_frontiera(foresta, X_sesso_train, y_sesso_train, "Random forest: 200 alberi profondi, a voto", ax=assi[1])
plt.show()

📌 La foresta è fatta degli **stessi alberi profondi** che da soli vanno in overfitting, eppure, mettendoli ai voti, fa meglio di un singolo albero profondo: è la **riduzione della varianza**. Si avvicina al livello dell'albero che abbiamo ottimizzato con `GridSearchCV`, ma **senza nessuna messa a punto** (*accurata senza tuning*). Il prezzo è l'interpretabilità: 200 alberi non si leggono come un `if`/`else`.

### 5.4 ✏️ Esercizio: quanti alberi servono?
Calcola l'accuracy media in cross-validation di una random forest con un numero crescente di alberi (`n_estimators`). Il grafico è già pronto.

In [ ]:
numero_alberi = [1, 2, 5, 10, 20, 50, 100, 200]
acc_foresta = []

for b in numero_alberi:
    # ✏️ crea una RandomForestClassifier con n_estimators=b e random_state=0,
    #    calcola l'accuracy media in cross-validation e aggiungila ad acc_foresta
    pass

plt.figure(figsize=(8, 4.5))
plt.plot(numero_alberi[:len(acc_foresta)], acc_foresta, marker="o")
plt.xscale("log"); plt.xticks(numero_alberi, numero_alberi)
plt.xlabel("numero di alberi (scala log)"); plt.ylabel("accuracy in cross-validation"); plt.grid(alpha=0.3)
plt.title("Più alberi: meglio, ma fino a un certo punto")
plt.show()

<details>
<summary><b>💡 Soluzione</b> (clicca per aprire, ma prima prova da solo!)</summary>

```python
numero_alberi = [1, 2, 5, 10, 20, 50, 100, 200]
acc_foresta = []

for b in numero_alberi:
    foresta_b = RandomForestClassifier(n_estimators=b, random_state=0)
    acc_foresta.append(cross_val_score(foresta_b, X_sesso_train, y_sesso_train, cv=cv).mean())

plt.figure(figsize=(8, 4.5))
plt.plot(numero_alberi[:len(acc_foresta)], acc_foresta, marker="o")
plt.xscale("log"); plt.xticks(numero_alberi, numero_alberi)
plt.xlabel("numero di alberi (scala log)"); plt.ylabel("accuracy in cross-validation"); plt.grid(alpha=0.3)
plt.title("Più alberi: meglio, ma fino a un certo punto")
plt.show()
```

</details>

📌 Dopo pochi alberi la curva smette di salire: le piccole oscillazioni che restano sono rumore della stima. Sono i **rendimenti decrescenti**: mediando $B$ alberi la varianza scende come $\rho\sigma^2 + (1-\rho)\sigma^2/B$. Il secondo termine sparisce aumentando $B$, il primo no: dipende da quanto gli alberi sono **correlati** tra loro ($\rho$). Oltre una certa soglia aggiungere alberi costa tempo e non migliora più.

### 5.5 Tutte le misure
Finora abbiamo usato solo il becco, per poter disegnare la frontiera. Ma abbiamo anche **pinna** e **peso**! Proviamo la foresta con tutte e quattro le misure: la random forest ci dice anche **quanto ha contato ogni feature** (`feature_importances_`).

> 🐍 `df.loc[indici, colonne]` seleziona righe per **etichetta** (l'indice a sinistra della tabella) e colonne per nome. Usando gli indici di `X_sesso_train` e `X_sesso_test` teniamo esattamente la stessa divisione.

In [ ]:
misure = ["becco_lunghezza", "becco_profondita", "pinna", "peso"]
X4_train, X4_test = df.loc[X_sesso_train.index, misure], df.loc[X_sesso_test.index, misure]  # stessa divisione di prima

foresta4 = RandomForestClassifier(n_estimators=200, random_state=0)
print(f"Cross-validation con 4 misure: {cross_val_score(foresta4, X4_train, y_sesso_train, cv=cv).mean():.3f}")
foresta4.fit(X4_train, y_sesso_train)

pd.Series(foresta4.feature_importances_, index=misure).sort_values().plot.barh(figsize=(7, 3), color="steelblue")
plt.xlabel("importanza"); plt.title("Quanto conta ogni misura per la foresta")
plt.tight_layout()
plt.show()

### 5.6 ⏩ Boosting
Il **boosting** fa l'opposto del bagging: invece di tanti alberi profondi e indipendenti, addestra **in sequenza** tanti alberi **poco profondi** (alto bias), e ognuno corregge gli errori di quelli precedenti. Riduce soprattutto il **bias**. Il `learning_rate` decide quanto conta ogni nuovo albero.

Mettiamo a confronto tutti i modelli in cross-validation, con le 4 misure:

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier

candidati = {
    "albero singolo (max_depth=4)": DecisionTreeClassifier(max_depth=4, random_state=0),
    "random forest (200 alberi)": RandomForestClassifier(n_estimators=200, random_state=0),
    "gradient boosting (100 alberi, max_depth=3)": GradientBoostingClassifier(
        n_estimators=100, learning_rate=0.1, max_depth=3, random_state=0),
}
righe = []
for nome, modello in candidati.items():
    punteggi = cross_val_score(modello, X4_train, y_sesso_train, cv=cv)
    righe.append({"modello": nome, "accuracy CV media": punteggi.mean(), "deviazione std": punteggi.std()})
pd.DataFrame(righe).round(3)

> 🤔 Con 4 misure i tre modelli sono praticamente alla pari: le differenze stanno dentro la deviazione standard. Il salto di qualità è venuto dall'**aggiungere informazione** (pinna e peso), non dal cambiare modello. Gli ensemble danno il meglio con dati più grandi e complessi di questi: sui dati tabellari il gradient boosting è spesso il modello da battere.

🔓 **La consegna finale.** Scegliamo la random forest, la più stabile (deviazione standard più bassa), la valutiamo **una volta** sul test set e rispondiamo di nuovo alla ricercatrice.

In [ ]:
print("Accuracy sul test set – random forest con 4 misure:", accuracy_score(y_sesso_test, foresta4.predict(X4_test)))

risposta["sesso stimato (foresta, 4 misure)"] = foresta4.predict(misteriosi[misure])
risposta

> 🤔 Le due risposte sono d'accordo su tutti i 9 pinguini? Su quali no? A quale daresti più fiducia, e perché?


### 🏠 Per casa (facoltativo)
1. Nella Parte 4.7 aggiungi `"criterion": ["gini", "entropy"]` alla griglia di `GridSearchCV`. Cambia qualcosa?
2. Usa i valori di `percorso.ccp_alphas` (Parte 4.6) come griglia per `ccp_alpha` in `GridSearchCV`: il post-pruning batte il pre-pruning?
3. Nella Parte 5.6 prova il gradient boosting con `learning_rate=0.01` e `n_estimators=500`. Migliora?
4. Usa una `RandomForestRegressor` per stimare il **peso** (Parte 2) con tutte le feature. Batte la regressione lineare?